In [1]:
import pandas as pd
import numpy as np
import os, json, time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


In [2]:
spark = (SparkSession.builder
    .appName("AmazonReviewsTraining")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} démarré")

your 131072x1 screen size is bogus. expect trouble
26/05/05 14:48:42 WARN Utils: Your hostname, LAPTOP-MPLRN9BK resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/05 14:48:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 14:48:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 3.5.0 démarré


In [3]:
print("Chargement train...")
train_df = spark.read.csv("../data/train.csv", 
                           header=True, 
                           inferSchema=True,
                           quote='"',
                           escape='"')

print("Chargement val...")
val_df = spark.read.csv("../data/val.csv", 
                         header=True, 
                         inferSchema=True,
                         quote='"',
                         escape='"')

# Cast sentiment en integer (sécurité)
from pyspark.sql.functions import col
train_df = train_df.withColumn("sentiment", col("sentiment").cast("integer"))
val_df   = val_df.withColumn("sentiment",   col("sentiment").cast("integer"))

# Vérifier
print(f"\nTrain: {train_df.count():,} rows")
print(f"Val:   {val_df.count():,} rows")

print("\nDistribution sentiment train:")
train_df.groupBy("sentiment").count().orderBy("sentiment").show()

Chargement train...


Chargement val...



Train: 454,763 rows
Val:   56,845 rows

Distribution sentiment train:


+---------+------+
|sentiment| count|
+---------+------+
|        0| 65630|
|        1| 34112|
|        2|355021|
+---------+------+



In [4]:
# Compter les classes
counts = train_df.groupBy("sentiment").count().collect()
count_dict = {row['sentiment']: row['count'] for row in counts}

print("Distribution originale:")
labels = {0: "negative", 1: "neutral", 2: "positive"}
for k in sorted(count_dict):
    print(f"  {labels[k]}: {count_dict[k]:,}")

neutral_count = count_dict[1]  # ~34,112

# Undersampling: positive → 3× neutral
target_positive = 3 * neutral_count
frac_positive = min(target_positive / count_dict[2], 1.0)

positive_df  = train_df.filter(col("sentiment") == 2).sample(False, frac_positive, seed=42)
negative_df  = train_df.filter(col("sentiment") == 0)
neutral_df   = train_df.filter(col("sentiment") == 1)

balanced = positive_df.union(negative_df).union(neutral_df)

print(f"\nAprès undersampling:")
balanced.groupBy("sentiment").count().orderBy("sentiment").show()

# Class weights
total = balanced.count()
new_counts = balanced.groupBy("sentiment").count().collect()
weights = {row['sentiment']: total / (3 * row['count']) for row in new_counts}

print(f"Class weights: {weights}")

balanced = balanced.withColumn("classWeight",
    when(col("sentiment") == 0, weights[0])
    .when(col("sentiment") == 1, weights[1])
    .otherwise(weights[2]))

print("✅ Dataset équilibré prêt")

Distribution originale:
  negative: 65,630
  neutral: 34,112
  positive: 355,021

Après undersampling:


+---------+------+
|sentiment| count|
+---------+------+
|        0| 65630|
|        1| 34112|
|        2|102306|
+---------+------+



Class weights: {2: 0.6583126437680423, 0: 1.0261973690893391, 1: 1.9743589743589745}
✅ Dataset équilibré prêt


In [5]:
from pyspark.sql.functions import col, when, coalesce, lit

# Nettoyer les nulls dans cleaned et sentiment
balanced = balanced.filter(
    col("cleaned").isNotNull() & 
    (col("cleaned") != "") &
    col("sentiment").isNotNull()
)

# Remplacer tout null restant par string vide (sécurité)
balanced = balanced.withColumn("cleaned", 
    coalesce(col("cleaned"), lit("")))

# Vérifier
print(f"Balanced après nettoyage: {balanced.count():,}")
print(f"Cleaned nulls: {balanced.filter(col('cleaned').isNull()).count()}")
print(f"Sentiment nulls: {balanced.filter(col('sentiment').isNull()).count()}")

# Faire pareil pour val_df
val_df = val_df.filter(
    col("cleaned").isNotNull() & 
    (col("cleaned") != "") &
    col("sentiment").isNotNull()
)
val_df = val_df.withColumn("cleaned", coalesce(col("cleaned"), lit("")))

print(f"\nVal après nettoyage: {val_df.count():,}")
print("✅ Prêt pour le training")

Balanced après nettoyage: 202,047
Cleaned nulls: 0


Sentiment nulls: 0

Val après nettoyage: 56,845
✅ Prêt pour le training


In [6]:
tokenizer = Tokenizer(inputCol="cleaned", outputCol="tokens")
remover   = StopWordsRemover(inputCol="tokens", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=20000)
idf = IDF(inputCol="rawFeatures", outputCol="features", minDocFreq=5)

print("✅ Pipeline TF-IDF défini")
print(f"  numFeatures: 20,000")

✅ Pipeline TF-IDF défini
  numFeatures: 20,000


In [7]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="sentiment",
    weightCol="classWeight",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0,
    family="multinomial"
)


lr_pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

# Grille d'hyperparamètres à tester
paramGrid = (ParamGridBuilder()
    .addGrid(lr.regParam,  [0.001, 0.01, 0.1])
    .addGrid(lr.maxIter,   [20, 50])
    .build())

# CrossValidator 3 folds (= 6 combinaisons × 3 = 18 entraînements)
cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="sentiment",
        predictionCol="prediction",
        metricName="f1"
    ),
    numFolds=3,
    seed=42
)

print("Entraînement LR avec CrossValidation... (~15-20 min)")
t0 = time.time()
lr_model = cv.fit(balanced)   # ← remplace lr_pipeline.fit(balanced)
t1 = time.time()
print(f"✅ LR (CV) entraîné en {(t1-t0)/60:.1f} min")

# Afficher les meilleurs hyperparamètres trouvés
best_lr = lr_model.bestModel.stages[-1]
print(f"Meilleur regParam : {best_lr._java_obj.getRegParam()}")
print(f"Meilleur maxIter  : {best_lr._java_obj.getMaxIter()}")

Entraînement LR avec CrossValidation... (~15-20 min)


✅ LR (CV) entraîné en 22.1 min
Meilleur regParam : 0.1
Meilleur maxIter  : 50


In [8]:
nb = NaiveBayes(
    featuresCol="features",
    labelCol="sentiment",
    smoothing=1.0,
    modelType="multinomial"
)

nb_pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, nb])

print("Entraînement NaiveBayes... (~1-2 min)")
t0 = time.time()
nb_model = nb_pipeline.fit(balanced)
t1 = time.time()
print(f"✅ NB entraîné en {(t1-t0)/60:.1f} min")

Entraînement NaiveBayes... (~1-2 min)


✅ NB entraîné en 0.7 min


In [9]:
def evaluate_model(model, name, df):
    preds = model.transform(df)
    
    f1_eval  = MulticlassClassificationEvaluator(
        labelCol="sentiment", predictionCol="prediction", metricName="f1")
    acc_eval = MulticlassClassificationEvaluator(
        labelCol="sentiment", predictionCol="prediction", metricName="accuracy")
    
    f1  = f1_eval.evaluate(preds)
    acc = acc_eval.evaluate(preds)
    
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"  F1-macro : {f1:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"{'='*40}")
    
    # Per-class stats
    preds.groupBy("sentiment", "prediction").count().orderBy("sentiment").show()
    
    return f1, preds

print("Évaluation sur validation set...")
f1_lr, preds_lr = evaluate_model(lr_model, "Logistic Regression", val_df)
f1_nb, preds_nb = evaluate_model(nb_model, "Naive Bayes", val_df)

Évaluation sur validation set...



  Logistic Regression
  F1-macro : 0.8341
  Accuracy : 0.8103


+---------+----------+-----+
|sentiment|prediction|count|
+---------+----------+-----+
|        0|       2.0|  498|
|        0|       0.0| 6356|
|        0|       1.0| 1349|
|        1|       1.0| 2723|
|        1|       2.0|  675|
|        1|       0.0|  866|
|        2|       1.0| 5252|
|        2|       2.0|36983|
|        2|       0.0| 2143|
+---------+----------+-----+




  Naive Bayes
  F1-macro : 0.7895
  Accuracy : 0.7589


+---------+----------+-----+
|sentiment|prediction|count|
+---------+----------+-----+
|        0|       2.0|  828|
|        0|       0.0| 5646|
|        0|       1.0| 1729|
|        1|       1.0| 2370|
|        1|       2.0|  926|
|        1|       0.0|  968|
|        2|       1.0| 5919|
|        2|       2.0|35121|
|        2|       0.0| 3338|
+---------+----------+-----+



In [11]:
if f1_lr >= f1_nb:
    best_model = lr_model
    best_name  = "LogisticRegression"
    best_f1    = f1_lr
    print(f"🏆 Meilleur modèle: LogReg (F1={f1_lr:.4f})")
else:
    best_model = nb_model
    best_name  = "NaiveBayes"
    best_f1    = f1_nb
    print(f"🏆 Meilleur modèle: NaiveBayes (F1={f1_nb:.4f})")

# Sauvegarder
os.makedirs("../models", exist_ok=True)
save_path = "../models/best_model"

print(f"Sauvegarde dans {save_path}...")
best_model.bestModel.write().overwrite().save(save_path)

best_lr_inner = best_model.bestModel.stages[-1]
print(f"Meilleur regParam : {best_lr_inner._java_obj.getRegParam()}")
print(f"Meilleur maxIter  : {best_lr_inner._java_obj.getMaxIter()}")

# Metadata
metadata = {
    "model_type": best_name,
    "f1_val": best_f1,
    "f1_lr": f1_lr,
    "f1_nb": f1_nb,
    "trained_on": str(pd.Timestamp.now()),
    "num_features": 20000,
    "train_size": train_df.count(),
    "balanced_size": balanced.count()
}

with open("../models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Modèle sauvegardé!")
print(json.dumps(metadata, indent=2))

🏆 Meilleur modèle: LogReg (F1=0.8341)
Sauvegarde dans ../models/best_model...
Meilleur regParam : 0.1
Meilleur maxIter  : 50


✅ Modèle sauvegardé!
{
  "model_type": "LogisticRegression",
  "f1_val": 0.83412131037336,
  "f1_lr": 0.83412131037336,
  "f1_nb": 0.7895392626025558,
  "trained_on": "2026-05-05 15:20:10.660193",
  "num_features": 20000,
  "train_size": 454763,
  "balanced_size": 202047
}
